<div style="background-color: #ADD8E6; border: 1px solid gray; padding: 3px">
    <h3>GraphRAG Data Generation </h3>
    The following is an overview of the workflow:
    <ul>
    <li>Uses a generic parser to parse out metadata for the following code components in each code file:
        <ul>
        <li>File Path</li>
        <li>Purpose</li>
        <li>Packages</li>
        <li>Classes</li>
        <li>Methods</li>
        <li>etc...</li>
        </ul>
    </li>
    <li>Adds the metadata for the retrieved components as comments to each code file.</li>
    </ul>
</div>

In [1]:
##############################################################################
# Imports
##############################################################################
from dotenv import load_dotenv
load_dotenv()
import nest_asyncio
nest_asyncio.apply()

### Prepare environment

In [2]:
def clone_from_repo(repo_url, destination_path, branch='master'):
    """
    Clones the given git repo to the specified destination.
    """
    ##############################################
    # Imports
    ##############################################
    from git import Repo
    import shutil
    import logging
    logging.basicConfig(level=logging.INFO)
    
    try:
        
        Repo.clone_from(repo_url, destination_path, branch=branch)
        
        logging.info(f"Repository '{repo_url}' cloned successfully to '{destination_path}'.")
        
    except Exception as e:
        
        logging.error(f"Error cloning repository: {e}")
        
        raise e

In [3]:
def prepare_environment(source_path: str, 
                        target_path: str,
                        git_repo: str,
                        git_branch: str):
    """ 
    Prepares the environment at the start of the pipeline.
    """
    ##############################################
    # Imports
    ##############################################
    import shutil
    import logging
    logging.basicConfig(level=logging.INFO)

    logging.info("Preparing the environment for pipeline run...")

    try:
        shutil.rmtree(source_path, ignore_errors=True)
        
        shutil.rmtree(target_path, ignore_errors=True)
        
        clone_from_repo(git_repo, source_path, branch=git_branch)
        
    except Exception as e:
        
        logging.error(f"Error deleting folders: {e}")
        
        raise e
    

### Generate dataset

In [ ]:
# ##############################################
# # Generate raw dataset
# ##############################################

def generate_raw_dataset(source_path: str,
                         target_path: str, 
                         git_repo: str,
                         language: str = "python",
                         split_sections=True,
                         config=False):
    """
    Walks source_path and collects all source files matching the given language's
    file extensions (or config file extensions when config=True).

    Returns a pandas DataFrame with columns: code, file_path, git_repo, language.
    Returns None if no matching files are found.
    """
    # ##############################################
    # # Imports
    # ##############################################
    from dotenv import load_dotenv
    import os
    load_dotenv()
    import pandas as pd
    from utils import code_utils
    import logging
    logging.basicConfig(level=logging.INFO)

    try:

        logging.info(f"Generating raw dataset for git repo={git_repo}, language={language}...")

        records = []

        excluded_dirs = code_utils.get_exclude_dirs_for_language(language)
    
        for root, dirs, files in os.walk(source_path):

            dirs[:] = [d for d in dirs if d not in excluded_dirs]

            if config:

                include_extensions = code_utils.get_config_file_extensions_for_language(language)

            else:

                include_extensions = code_utils.get_file_extensions_for_language(language)
            
            for filename in files:
            
                if os.path.splitext(filename)[1] not in include_extensions:
                    
                    continue
    
                logging.debug(f"Processing {filename}...")
                
                abs_path = os.path.join(root, filename)

                if code_utils.is_large_code_file(abs_path, max_size=200_000):

                    code_utils.process_large_code_file(abs_path, source_path)

                    continue
                
                rel_path = os.path.relpath(abs_path, source_path)
                
                try:
                    
                    with open(abs_path, "r", encoding="utf-8") as f:
                        
                        code = f.read()
    
                        records.append({"code":code,
                                        "file_path": rel_path,
                                        "git_repo": git_repo,
                                        "language": language})
                        
                except (UnicodeDecodeError, PermissionError):
                    continue
    
        df = pd.DataFrame(records) if records else None
    
        return df
        
    except Exception as e:
        
        logging.error(f"Error generating dataframe with raw code: {e}")

        raise e

            

In [5]:
# ##############################################
# # Extract metadata from code dataset
# ##############################################

def get_parsed_code_metadata(df, language, config=False):
    """ Given a dataframe with file paths and code content, generates a dataset that includes metadata for each file."""

    # ##############################################
    # # Imports
    # ##############################################
    from datasets import Dataset
    from sdg_hub.core.flow import Flow
    from flows.flow_extensions import CustomDeleteColumnsBlock
    from datetime import datetime
    import logging
    import os
    logging.basicConfig(level=logging.INFO)

    try:

        logging.info("Parsing code metadata...")
        
        dataset = Dataset.from_pandas(df)

        flow_path = "flows/config_generation/flow.yaml" if config else "flows/code_generation/flow.yaml"

        flow = Flow.from_yaml(flow_path)
        
        flow.set_model_config(
            model=f"{os.getenv('GRAPHRAG_LLM_PROVIDER')}/{os.getenv('GRAPHRAG_LLM_ID')}",
            
            api_base=os.getenv("GRAPHRAG_LLM_API_BASE"),
            
            api_key=os.getenv("GRAPHRAG_LLM_TOKEN"),
            
            temperature=0,
            
            max_tokens = 32_000,
            
            response_format={"type": "json_object"},
            
            top_k=1,
        )
    
        converted_dataset = flow.generate(dataset, max_concurrency=10)
        
        converted_df = converted_dataset.to_pandas()

        converted_df.to_csv(f"data_{language}_{'config_' if config else '_'}{str(int(datetime.now().timestamp()))}.csv")

        return converted_df
        
    except Exception as e:
        
        logging.error(f"Error extracting metadata from code: {e}")

        raise e

In [ ]:
# ##############################################
# # Generate code comment from metadata
# ##############################################
def generate_code_comment(metadata: dict, file_path: str, config=False):
    """
    Builds a structured text comment from a code file's metadata dictionary.

    The comment includes details such as the file location, package, purpose,
    imports, classes, functions, and methods. It is wrapped with the
    appropriate begin/end comment delimiters for the file's language (e.g.
    /* */ for Java, \"\"\" for Python).
    When config=True, uses config-file comment delimiters instead of source-file ones.
    """
    # ##############################################
    # # Imports
    # ##############################################
    import os
    from utils import code_utils
    import logging
    logging.basicConfig(level=logging.INFO)

    try:
    
        lines = []
        
        lines.append(f"This file is located at {metadata.get('file_path')} from repository {metadata.get('git_repo')}.")
        
        if metadata.get('package'):
            
            lines.append(f"\n Package: {metadata['package']}")
    
        if metadata.get('purpose'):
            
            lines.append(f"\n Purpose: {metadata['purpose']}")
        
        if metadata.get('imports'):
            
            lines.append(f"\nDependencies:")
                
            lines.extend(f"- {file_path} imports {imp}" for imp in metadata['imports'])
        
        if metadata.get('classes'):
            
            lines.append(f"\n Classes:")
            
            lines.extend(f"- {cls}" for cls in metadata['classes'])
        
        if metadata.get('functions'):
            
            lines.append(f"\n Functions:")
            
            lines.extend([f"- {func}" for func in metadata['functions']])
        
        if metadata.get('methods'):
            
            lines.append(f"\n Methods:")
            
            lines.extend([f"- {func}" for func in metadata['methods']])

        if config:

            extension = os.path.splitext(file_path)[1]

            begin, end = code_utils.get_comment_delimiters_for_file_extension(extension)

        else:

            begin, end = code_utils.get_comment_delimiters_for_language(metadata.get('language'))
        
        return begin + '\n'.join(lines) + end
              
    except Exception as e:
        
        logging.error(f"Error generating comment: {e}")

        raise e
    
    

In [ ]:
# ##############################################
# # Save generated code and metadata to target dir
# ##############################################
def save_code_and_metadata_files(df, target_path, config=False):
    """
    Iterates over a DataFrame of code files and their extracted metadata,
    writing two output files per row into target_path:

    - <file_path>.txt            : the original code prefixed with a generated comment header
    - <file_path>_metadata.txt   : a flattened plain-text representation of the metadata

    Rows for which no parseable metadata is found are skipped.
    """
    # ##############################################
    # # Imports
    # ##############################################
    import os
    from pathlib import Path          
    import logging
    import json 
    from utils import json_utils
    logging.basicConfig(level=logging.INFO)     

    try:
        logging.info("Saving code and metadata files...")
        
        for _, row in df.iterrows():   

            code = row["code"]
            
            metadata = json_utils.extract_json_from_string(row["extracted_data"])

            if not metadata:
                logging.info(f"No metadata found for file {row['file_path']}. Skipping...")

                continue
            
            rel_file_path = row["file_path"]

            target_file_path = os.path.join(target_path, Path(rel_file_path).with_suffix(".txt"))

            metadata_file_path = os.path.join(target_path, str(Path
                                                               (rel_file_path).with_suffix("")) + "_metadata.txt")

            code_header_comment = generate_code_comment(metadata=metadata,
                                                        file_path=rel_file_path, config=config) or ""
            
            os.makedirs(os.path.dirname(target_file_path), exist_ok=True)   
            
            os.makedirs(os.path.dirname(metadata_file_path), exist_ok=True)   
            
            with open(target_file_path, "w", encoding="utf-8") as f:
                
                f.write(f"{code_header_comment}\n{code}")  
            
            with open(metadata_file_path, "w", encoding="utf-8") as f:

                flattened_metadata = json_utils.flatten_code_metadata(metadata)
                
                f.write(flattened_metadata)
              
    except Exception as e:
        
        logging.error(f"Error saving code and metadata: {e}")

        raise e
    

### Run full pipeline


In [8]:
#########################################################################
# Driver method for generating metadata
#########################################################################
def generate_code_and_meta(git_repo: str, git_branch: str, language: str, source_path: str, target_path: str, config: bool = False):
    """ Runs the full pipeline. """

    # ##############################################
    # # Imports
    # ##############################################
    import logging
    logging.basicConfig(level=logging.INFO)
    
    try:

        code_df = generate_raw_dataset(source_path, target_path, git_repo, language=language, config=config)

        if code_df is None:

            logging.info(f"No {language} files found based on specified extensions (config={config}).")

            return

        code_and_metadata_df = get_parsed_code_metadata(code_df, language=language, config=config)

        save_code_and_metadata_files(code_and_metadata_df, target_path, config=config)
        
        logging.info(f"Successfully generated code metadata for '{git_repo}'.")
        
    except Exception as e:
        
        logging.error(f"Error generating code and metadata: {e}")

        raise e


In [ ]:
#########################################################################
# Instance Variables
#########################################################################
_LANGUAGES = ["java", "shell", "sql"]
_GIT_REPO = "https://github.com/agapebondservant/spacefx-sample-javafx.git"
_GIT_BRANCH = "main"
_SOURCE_PATH = "source"
_TARGET_PATH = "target"

In [ ]:
#########################################################################
# Imports
#########################################################################
import logging
logging.basicConfig(level=logging.INFO)
import traceback


prepare_environment(source_path=_SOURCE_PATH, target_path=_TARGET_PATH, git_repo=_GIT_REPO, git_branch=_GIT_BRANCH)

try:

    for language in _LANGUAGES:

        ######################################################################
        # Source code
        ######################################################################

        logging.info(f"Starting source code data generation pipeline with language={language}...")

        generate_code_and_meta(git_repo=_GIT_REPO,
                               git_branch=_GIT_BRANCH,
                               language=language,
                               source_path=_SOURCE_PATH,
                               target_path=_TARGET_PATH,
                               config=False)

        logging.info(f"Source code data generation pipeline with language={language} complete.")

        ######################################################################
        # Config files
        ######################################################################

        logging.info(f"Starting config data generation pipeline with language={language}...")

        generate_code_and_meta(git_repo=_GIT_REPO,
                               git_branch=_GIT_BRANCH,
                               language=language,
                               source_path=_SOURCE_PATH,
                               target_path=_TARGET_PATH,
                               config=True)

        logging.info(f"Config data generation pipeline with language={language} complete.")

        
except Exception as e:

    logging.error("PIPELINE FAILED!")

    logging.error(traceback.format_exc())